### Overview
##### Welcome to the 2026 Kaggle Playground Series! We plan to continue in the spirit of previous playgrounds, providing interesting and approachable datasets for our community to practice their machine learning skills, and anticipate a competition each month.

##### Goal: Predict whether a Formula 1 driver will pit on the next lap.

In [44]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as pt
import seaborn as sns


Load Dataset

In [45]:
df=pd.read_csv('train.csv')
df.head()


,id,Driver,Compound,Race,Year,PitStop,LapNumber,Stint,TyreLife,Position,LapTime (s),LapTime_Delta,Cumulative_Degradation,RaceProgress,Position_Change,PitNextLap
0,0,D109,HARD,Canadian Grand Prix,2022,0,50,2,39.0,8,78.491,-7.564,21.019,0.714286,5.0,1.0
1,1,D086,HARD,Dutch Grand Prix,2025,1,27,2,7.0,4,75.095,-32.617,-223.207,0.346154,-3.0,0.0
2,2,ZON,HARD,Austrian Grand Prix,2022,0,59,3,22.0,13,70.945,-7.540,-100.529,0.819444,3.0,1.0
3,3,SPE,MEDIUM,Pre-Season Testing,2023,0,2,1,2.0,7,94.361,-7.324,-7.324,0.076923,0.0,0.0
4,4,D019,HARD,Azerbaijan Grand Prix,2022,1,26,3,6.0,2,107.878,8.965,-14.139,0.361111,3.0,0.0


Exploratory Data Analysis

In [46]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 439140 entries, 0 to 439139
Data columns (total 16 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   id                      439140 non-null  int64  
 1   Driver                  439140 non-null  object 
 2   Compound                439140 non-null  object 
 3   Race                    439140 non-null  object 
 4   Year                    439140 non-null  int64  
 5   PitStop                 439140 non-null  int64  
 6   LapNumber               439140 non-null  int64  
 7   Stint                   439140 non-null  int64  
 8   TyreLife                439140 non-null  float64
 9   Position                439140 non-null  int64  
 10  LapTime (s)             439140 non-null  float64
 11  LapTime_Delta           439140 non-null  float64
 12  Cumulative_Degradation  439140 non-null  float64
 13  RaceProgress            439140 non-null  float64
 14  Position_Change     

In [47]:
df.describe()

,id,Year,PitStop,LapNumber,Stint,TyreLife,Position,LapTime (s),LapTime_Delta,Cumulative_Degradation,RaceProgress,Position_Change,PitNextLap
count,439140.000000,439140.000000,439140.000000,439140.000000,439140.000000,439140.000000,439140.000000,439140.000000,439140.000000,439140.000000,439140.000000,439140.000000,439140.000000
mean,219569.500000,2023.523544,0.136118,23.105909,1.789113,14.158231,9.630339,90.948735,-3.770040,-25.721759,0.337661,0.101542,0.198982
std,126768.942943,1.024930,0.342915,16.958261,0.950194,9.801338,5.278770,19.772769,43.945759,54.766573,0.253277,4.006765,0.399235
min,0.000000,2022.000000,0.000000,1.000000,1.000000,1.000000,1.000000,67.694000,-2403.895000,-274.564000,0.012821,-18.000000,0.000000
25%,109784.750000,2023.000000,0.000000,9.000000,1.000000,6.000000,5.000000,82.621000,-8.884000,-46.566250,0.129870,-1.000000,0.000000
50%,219569.500000,2024.000000,0.000000,19.000000,2.000000,12.000000,10.000000,90.521000,-0.295000,-20.994000,0.269231,0.000000,0.000000
75%,329354.250000,2024.000000,0.000000,36.000000,2.000000,20.000000,14.000000,98.471000,0.115000,-6.199000,0.513158,2.000000,0.000000
max,439139.000000,2025.000000,1.000000,78.000000,8.000000,77.000000,20.000000,2507.607000,2423.932000,2412.026000,1.000000,18.000000,1.000000


In [48]:
df.columns

Index(['id', 'Driver', 'Compound', 'Race', 'Year', 'PitStop', 'LapNumber',
       'Stint', 'TyreLife', 'Position', 'LapTime (s)', 'LapTime_Delta',
       'Cumulative_Degradation', 'RaceProgress', 'Position_Change',
       'PitNextLap'],
      dtype='object')


- **id**: Unique identifier for each lap record.  
- **Driver**: Name or code of the Formula 1 driver.  
- **Compound**: Tyre compound used (Soft, Medium, Hard, etc.).  
- **Race**: Name or identifier of the race.  
- **Year**: Year in which the race took place.  
- **PitStop**: Number of pit stops completed so far.  
- **LapNumber**: Current lap number in the race.  
- **Stint**: Stint number (period between pit stops).  
- **TyreLife**: Age of the tyre in laps since last pit stop.  
- **Position**: Driver’s current race position.  
- **LapTime (s)**: Time taken to complete the lap (in seconds).  
- **LapTime_Delta**: Difference compared to the previous lap time.  
- **Cumulative_Degradation**: Measure of tyre wear accumulated over laps.  
- **RaceProgress**: Percentage of race completed (0–100%).  
- **Position_Change**: Change in race position compared to the previous lap.  
- **PitNextLap**: Target variable → 1 if the driver pits on the next lap, 0 otherwise.

In [49]:
Y=df.PitNextLap
Y

0         1.0
1         0.0
2         1.0
3         0.0
4         0.0
         ... 
439135    0.0
439136    0.0
439137    0.0
439138    0.0
439139    0.0
Name: PitNextLap, Length: 439140, dtype: float64

In [50]:
from sklearn.preprocessing import LabelEncoder
le=LabelEncoder()
## Encode Categorical Variables
df.Compound=le.fit_transform(df.Compound)


In [55]:
## scale continous features
from sklearn.preprocessing import StandardScaler
sc=StandardScaler()
df['LapTime (s)']=sc.fit_transform(df[['LapTime (s)']])
df['LapTime_Delta']=sc.fit_transform(df[['LapTime_Delta']])
df['Cumulative_Degradation']=sc.fit_transform(df[['Cumulative_Degradation']])

df.describe()

,id,Compound,Year,PitStop,LapNumber,Stint,TyreLife,Position,LapTime (s),LapTime_Delta,Cumulative_Degradation,RaceProgress,Position_Change,PitNextLap
count,439140.000000,439140.000000,439140.000000,439140.000000,439140.000000,439140.000000,439140.000000,439140.000000,4.391400e+05,4.391400e+05,4.391400e+05,439140.000000,439140.000000,439140.000000
mean,219569.500000,1.278217,2023.523544,0.136118,23.105909,1.789113,14.158231,9.630339,2.621212e-18,2.912458e-18,-5.468949e-18,0.337661,0.101542,0.198982
std,126768.942943,1.082766,1.024930,0.342915,16.958261,0.950194,9.801338,5.278770,1.000001e+00,1.000001e+00,1.000001e+00,0.253277,4.006765,0.399235
min,0.000000,0.000000,2022.000000,0.000000,1.000000,1.000000,1.000000,1.000000,-1.176100e+00,-5.461568e+01,-4.543694e+00,0.012821,-18.000000,0.000000
25%,109784.750000,0.000000,2023.000000,0.000000,9.000000,1.000000,6.000000,5.000000,-4.211724e-01,-1.163699e-01,-3.806065e-01,0.129870,-1.000000,0.000000
50%,219569.500000,2.000000,2024.000000,0.000000,19.000000,2.000000,12.000000,10.000000,-2.163253e-02,7.907576e-02,8.632573e-02,0.269231,0.000000,0.000000
75%,329354.250000,2.000000,2024.000000,0.000000,36.000000,2.000000,20.000000,14.000000,3.804360e-01,8.840546e-02,3.564726e-01,0.513158,2.000000,0.000000
max,439139.000000,4.000000,2025.000000,1.000000,78.000000,8.000000,77.000000,20.000000,1.222217e+02,5.524321e+01,4.451165e+01,1.000000,18.000000,1.000000


Feature Engineering
